# Lesson 1.1 — DevOps Environment Setup

**Level: Beginner**

## Learning Objectives
- Create and activate Python virtual environments
- Understand the recommended project layout for ops automation scripts
- Manage dependencies with `pip`, `requirements.txt`, and `pyproject.toml`
- Verify your environment is ready for the course

**Produces:** `ops-platform/` project skeleton with all dependencies installed.

## Concepts

### Why Virtual Environments?

A **virtual environment** gives every project its own isolated Python interpreter and package set.
Without it, installing `boto3==1.30` for one project can silently break another that needs `boto3==1.26`.

```
project-a/.venv/  ← boto3==1.30, requests==2.31
project-b/.venv/  ← boto3==1.26, requests==2.28   (no conflict)
```

Create and activate with:
```bash
python -m venv .venv
source .venv/bin/activate        # Linux / macOS
.venv\Scripts\activate           # Windows
```

### Recommended Project Layout
```
ops-platform/
├── .venv/                  # virtual environment (never commit)
├── src/ops_platform/       # source code
│   ├── __init__.py
│   ├── cloud/              # AWS / Azure / GCP modules
│   ├── security/           # DevSecOps modules
│   ├── monitoring/         # Prometheus / OTel / alerting
│   └── cli.py              # Click CLI entry point
├── tests/
├── configs/
├── .env                    # secrets (NEVER commit)
├── .env.example            # safe template (commit this)
├── requirements.txt        # pinned production deps
└── pyproject.toml          # package metadata & entry points
```

*Reference: [Python Packaging User Guide](https://packaging.python.org/)*

In [ ]:
# Step 1 — verify Python version and virtual-environment status
import sys, subprocess

print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")

result = subprocess.run([sys.executable, "-m", "pip", "--version"],
                        capture_output=True, text=True)
print(f"pip: {result.stdout.strip()}")

in_venv = sys.prefix != sys.base_prefix
print(f"\nRunning inside venv: {in_venv}")
if not in_venv:
    print("  ⚠  Recommendation: activate a virtual environment first.")
    print("     python -m venv .venv && source .venv/bin/activate  (Linux/macOS)")
    print("     python -m venv .venv && .venv\\Scripts\\activate   (Windows)")
else:
    print(f"  ✓  venv path: {sys.prefix}")

## Code-Along: Step 1 — Scaffold the Project

In [ ]:
from pathlib import Path

root = Path("ops-platform")

dirs = [
    root / "src" / "ops_platform" / "cloud",
    root / "src" / "ops_platform" / "security",
    root / "src" / "ops_platform" / "monitoring",
    root / "tests",
    root / "configs",
]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    (d / "__init__.py").touch()          # marks as Python package
    print(f"  created {d}")

print("\n✓ Directory tree created.")

## Code-Along: Step 2 — Create Bootstrap Files

In [ ]:
# .env.example — safe template (commit this, never .env itself)
(root / ".env.example").write_text(
    "# Copy to .env and fill in real values.  NEVER commit .env\n"
    "AWS_ACCESS_KEY_ID=\n"
    "AWS_SECRET_ACCESS_KEY=\n"
    "AWS_DEFAULT_REGION=us-east-1\n"
    "SLACK_WEBHOOK_URL=\n"
    "LOG_LEVEL=INFO\n"
)

# requirements.txt
(root / "requirements.txt").write_text(
    "boto3>=1.28.0\n"
    "python-dotenv>=1.0.0\n"
    "pyyaml>=6.0.1\n"
    "click>=8.1.7\n"
    "rich>=13.7.0\n"
    "structlog>=24.1.0\n"
    "prometheus_client>=0.19.0\n"
    "requests>=2.31.0\n"
)

# .gitignore
(root / ".gitignore").write_text(
    ".venv/\n__pycache__/\n*.pyc\n.env\ndist/\nbuild/\n"
)

print("✓ Bootstrap files written:")
for f in sorted(root.rglob("*")):
    if f.is_file():
        print(f"  {f}")

## Code-Along: Step 3 — Install & Verify Dependencies

In [ ]:
import subprocess, sys, importlib

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "boto3", "python-dotenv", "pyyaml", "click", "rich", "structlog",
     "prometheus_client", "requests", "--quiet"],
    capture_output=True
)

packages = [
    ("boto3",             "AWS SDK"),
    ("dotenv",            "python-dotenv"),
    ("yaml",              "PyYAML"),
    ("click",             "Click CLI"),
    ("rich",              "Rich terminal output"),
    ("structlog",         "Structured logging"),
    ("prometheus_client", "Prometheus client"),
    ("requests",          "HTTP library"),
]

print("Dependency verification:")
all_ok = True
for pkg, desc in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "installed")
        print(f"  ✓  {desc:<28} {ver}")
    except ImportError:
        print(f"  ✗  {desc}: NOT INSTALLED")
        all_ok = False

print("\n✓ All dependencies ready." if all_ok else "\n✗ Some packages missing.")

## Exercise

Extend the scaffold with a `monitoring/` sub-module.

**Your task:**
1. Create `ops-platform/src/ops_platform/monitoring/`
2. Add `__init__.py` with docstring `"""Monitoring and observability."""`
3. Add `prometheus_client>=0.19.0` to `requirements.txt`
4. Print the contents of `requirements.txt` to verify

In [ ]:
# TODO: create the monitoring directory
# TODO: write __init__.py
# TODO: append prometheus_client to requirements.txt
# TODO: print requirements.txt


## Key Takeaways
- Always use virtual environments — prevents dependency conflicts between projects
- `.env` for secrets, `.env.example` as the committed template
- `requirements.txt` is the standard for reproducible installs in ops scripts
- Structure source code under `src/` to avoid import shadowing

---
**Next:** [Lesson 1.2 — OS & Process Automation](lesson-1.2-os-process-automation.ipynb)